<a href="https://colab.research.google.com/github/bhavikd-ai/rag/blob/main/Re_Ranking_in_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# INSTALLING REQUIRE PACKAGES

! pip install sentence-transformers torch numpy -q

In [2]:
# IMPORTING REQUIRE LIBRARIES

from sentence_transformers import SentenceTransformer
from sentence_transformers import CrossEncoder
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

In [ ]:
# INITIALIZE ENCODER

encoder = SentenceTransformer('all-MiniLM-L6-v2')

# INITIALIZE RE-RANKER
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

In [5]:
# SAMPLE DOCUMENTS

documents = [
    "Python is a programming language for data science",
    "Machine learning uses algorithms to learn from data",
    "Python and data science are popular together",
    "The Eiffel Tower is in Paris"
]

In [6]:
# GENERATE EMBEDDINGS

doc_embeddings = encoder.encode(documents)

In [7]:
# EXTRACT TOP - K

query = "python data science"

query_embedding = encoder.encode([query])

similarities = cosine_similarity(query_embedding, doc_embeddings)[0]

top_k_indices = np.argsort(similarities)[-3:][::-1]

In [8]:
#CREATE A DATAFRAME WITH SIMILARITY SEARCH SCORE

df = pd.DataFrame()
df["documents"] = documents
df["similarities"] = similarities
df["query"] = query
df.sort_values(by="similarities", ascending=False, inplace = True)
df.reset_index(drop=True)
df

,documents,similarities,query
0,Python is a programming language for data science,0.795557,python data science
2,Python and data science are popular together,0.742293,python data science
1,Machine learning uses algorithms to learn from...,0.316420,python data science
3,The Eiffel Tower is in Paris,-0.028011,python data science


In [9]:
#INITIAL RETRIEVAL

initial_candidates = df["documents"].to_list()

In [10]:
#QUERY DOCUMENT PAIRING

pairs = [[query, doc] for doc in initial_candidates]

pairs

[['python data science', 'Python is a programming language for data science'],
 ['python data science', 'Python and data science are popular together'],
 ['python data science',
  'Machine learning uses algorithms to learn from data'],
 ['python data science', 'The Eiffel Tower is in Paris']]

In [11]:
#RE-RANKING

rerank_scores = reranker.predict(pairs)

rerank_scores

array([  8.446303,   7.433578, -10.772228, -11.391971], dtype=float32)

In [12]:
# ADDING RE-RANKING IN DF

df["re-rank score"] = rerank_scores

In [13]:
#VIEW THE DATAFRAME

df

,documents,similarities,query,re-rank score
0,Python is a programming language for data science,0.795557,python data science,8.446303
2,Python and data science are popular together,0.742293,python data science,7.433578
1,Machine learning uses algorithms to learn from...,0.316420,python data science,-10.772228
3,The Eiffel Tower is in Paris,-0.028011,python data science,-11.391971


In [16]:
# RERANK FUNCTION

def dense_retrieval_vs_rerank(query, documents, top_k):
  doc_embeddings = encoder.encode(documents)

  query_embedding = encoder.encode([query])

  similarities = cosine_similarity(query_embedding, doc_embeddings)[0]

  top_k_indices = np.argsort(similarities)[-top_k:][::-1]

  df = pd.DataFrame()
  df["documents"] = documents
  df["similarities"] = similarities
  df["query"] = query
  df.sort_values(by="similarities", ascending=False, inplace = True)
  df.reset_index(drop=True)

  initial_candidates = df["documents"].to_list()

  pairs = [[query, doc] for doc in initial_candidates]

  rerank_scores = reranker.predict(pairs)

  df["re-rank score"] = rerank_scores

  return df

In [18]:
# TEST

documents = [
    "To repair a dripping tap, first turn off water supply, then replace the washer.",
    "The Eiffel Tower is located in Paris, France and was built in 1889.",
    "Fix a leaky faucet by checking the O-ring and valve seat for damage.",
    "Best pasta recipes for dinner include carbonara and bolognese.",
    "Turn off water valves under sink, disassemble faucet handle, replace cartridge."
]

query = "How to fix a leaky faucet?"


df2 = dense_retrieval_vs_rerank(query, documents, top_k=4)

In [20]:
df2.sort_values(by="re-rank score", ascending=False, inplace = True)
df2.reset_index(drop=True, inplace=True)

In [21]:
df2

,documents,similarities,query,re-rank score
0,Fix a leaky faucet by checking the O-ring and ...,0.815416,How to fix a leaky faucet?,8.299855
1,"To repair a dripping tap, first turn off water...",0.532385,How to fix a leaky faucet?,1.567336
2,"Turn off water valves under sink, disassemble ...",0.538304,How to fix a leaky faucet?,-0.463815
3,"The Eiffel Tower is located in Paris, France a...",0.012739,How to fix a leaky faucet?,-11.274267
4,Best pasta recipes for dinner include carbonar...,0.046567,How to fix a leaky faucet?,-11.296608
